# Week 1: ML Fundamentals and Data Preprocessing

## Titanic Survival Prediction — Data Cleaning

This notebook covers dataset loading and summary statistics, missing-value imputation, categorical encoding, an age visualization, and a train/test split example. The mini project exports the cleaned passenger table to `data/Titanic_Cleaned.csv`.

**ML in brief:** Machine learning uses data to learn patterns for predictions or decisions. Supervised learning learns from labeled examples (such as survival), unsupervised learning finds structure without labels (such as passenger groups), and reinforcement learning learns actions from rewards and penalties.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "Titanic-Dataset.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "Titanic-Dataset.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "Titanic_Cleaned.csv"

## Assignment 1 — Load and explore the dataset

Use `.info()` to inspect column types and non-null counts, `.describe()` for numeric summary statistics, and a null count to identify columns that need cleaning.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
df.info()
display(df.describe())
display(df.isna().sum().rename("missing values"))

## Assignment 2 — Handle missing values

Mean and median are common numeric imputation choices. The small example below shows both approaches; for the Titanic `Age` and `Fare` columns, use the median because it is less affected by extreme values. Fill the categorical `Embarked` column with its mode. `Cabin` is dropped because most entries are missing.

In [ ]:
numeric_example = pd.Series([10.0, 12.0, None, 100.0])
mean_filled = numeric_example.fillna(numeric_example.mean())
median_filled = numeric_example.fillna(numeric_example.median())
print("Mean-imputed example:", mean_filled.tolist())
print("Median-imputed example:", median_filled.tolist())

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode().iloc[0])
df = df.drop(columns=["Cabin"])
print("Remaining missing values:")
display(df.isna().sum())

## Assignment 3 — Encode categorical variables

`LabelEncoder` maps the two `Sex` categories to integers. `OneHotEncoder` creates indicator columns for the `Embarked` categories. These encoders are fit here to demonstrate preprocessing on this dataset; for model evaluation, fit preprocessing on training data only.

In [ ]:
sex_encoder = LabelEncoder()
df["Sex"] = sex_encoder.fit_transform(df["Sex"])
print("Sex mapping:", dict(zip(sex_encoder.classes_, sex_encoder.transform(sex_encoder.classes_))))

embarked_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype="int64")
embarked_values = embarked_encoder.fit_transform(df[["Embarked"]])
embarked_columns = embarked_encoder.get_feature_names_out(["Embarked"])
embarked_df = pd.DataFrame(embarked_values, columns=embarked_columns, index=df.index)
df = pd.concat([df.drop(columns=["Embarked"]), embarked_df], axis=1)
display(df.head())

## Visualize age distribution

The plot shows passenger ages after median imputation.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["Age"], bins=30, kde=True, color="skyblue")
plt.title("Age Distribution of Titanic Passengers")
plt.xlabel("Age (years)")
plt.ylabel("Number of passengers")
plt.tight_layout()
plt.show()

## Train/test split example

This is a demonstration using `Survived` as the target. The split is performed after cleaning for simplicity; in a production modeling workflow, split first and fit imputation and encoding on the training data only to avoid data leakage.

In [ ]:
X = df.drop(columns=["Survived"])
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train features:", X_train.shape, "Test features:", X_test.shape)

## Export the cleaned dataset

The CSV includes the `Survived` label because it is a cleaned version of the source data, not a model input matrix.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df):,} rows and {len(df.columns)} columns to {OUTPUT_PATH}")